In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler

#Base Clasificación Random forest.

train_data = pd.read_csv('./data/train.csv')
test_data = pd.read_csv('./data/test.csv')

# Exploración rápida de los datos
print(train_data.head())
print(train_data.info())

#Feature engineering

# Crear X e Y 
X = train_data.drop(['ID', 'SeriousDlqin2yrs'], axis=1)  # Excluir ID y la columna objetivo
y = train_data['SeriousDlqin2yrs']  # Columna objetivo

# Dividir los datos en conjuntos de entrenamiento y validación
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar las características 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
test_data_scaled = scaler.transform(test_data.drop('ID', axis=1))

# Entrenar el modelo con los hiperparámetros especificados en el GCV
model = RandomForestClassifier(random_state=42, n_jobs=-1,
                               n_estimators=800, max_depth=12,
                               max_leaf_nodes=89, max_features=5,
                               min_samples_split=11,
                               min_samples_leaf=1,
                               min_impurity_decrease=0.0)
model.fit(X_train, y_train)

# Evaluar el modelo en el conjunto de entrenamiento y validación
train_accuracy = model.score(X_train, y_train)
val_accuracy = model.score(X_val, y_val)
print(f'Accuracy en el conjunto de entrenamiento: {train_accuracy:.4f}')
print(f'Accuracy en el conjunto de validación: {val_accuracy:.4f}')

# Calcular el AUC-ROC en el conjunto de validación
y_val_proba = model.predict_proba(X_val)[:, 1]  # Probabilidades de la clase positiva
val_auc_roc = roc_auc_score(y_val, y_val_proba)
print(f'AUC-ROC en el conjunto de validación: {val_auc_roc:.4f}')

# Predecir en el conjunto de prueba
test_predictions = model.predict_proba(test_data_scaled)[:, 1]  # Probabilidades de la clase positiva

# Guardar las predicciones en un archivo CSV para enviar a Kaggle
submission = pd.DataFrame({'ID': test_data['ID'], 'SeriousDlqin2yrs': test_predictions})
submission.to_csv('borja.csv', index=False)

print("¡Predicciones guardadas en borja.csv!, a ganar")

       ID  RevolvingUtilizationOfUnsecuredLines  Age  \
0    9580                              0.668999   58   
1   39755                              0.015922   71   
2  118799                              0.183062   52   
3   16489                              0.162301   77   
4  149857                              0.404199   30   

   NumberOfTime30-59DaysPastDueNotWorse  DebtRatio  MonthlyIncome  \
0                                     2   0.449504         3425.0   
1                                     0   6.000000            NaN   
2                                     1   0.035593         5000.0   
3                                     0   0.227886         2000.0   
4                                     0   0.026010         5843.0   

   NumberOfOpenCreditLinesAndLoans  NumberOfTimes90DaysLate  \
0                                9                        1   
1                                5                        0   
2                                9                        0